In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [3]:
dataset_name = 'HuggingFaceTB/smoltalk2'
ds = load_dataset(dataset_name, 'SFT')

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/105 [00:00<?, ?it/s]

In [4]:
ds.keys()

dict_keys(['LongAlign_64k_Qwen3_32B_yarn_131k_think', 'OpenThoughts3_1.2M_think', 'aya_dataset_Qwen3_32B_think', 'multi_turn_reasoning_if_think', 's1k_1.1_think', 'smolagents_toolcalling_traces_think', 'smoltalk_everyday_convs_reasoning_Qwen3_32B_think', 'smoltalk_multilingual8_Qwen3_32B_think', 'smoltalk_systemchats_Qwen3_32B_think', 'table_gpt_Qwen3_32B_think', 'LongAlign_64k_context_lang_annotated_lang_6_no_think', 'Mixture_of_Thoughts_science_no_think', 'OpenHermes_2.5_no_think', 'OpenThoughts3_1.2M_no_think_no_think', 'hermes_function_calling_v1_no_think', 'smoltalk_multilingual_8languages_lang_5_no_think', 'smoltalk_smollm3_everyday_conversations_no_think', 'smoltalk_smollm3_explore_instruct_rewriting_no_think', 'smoltalk_smollm3_smol_magpie_ultra_no_think', 'smoltalk_smollm3_smol_rewrite_no_think', 'smoltalk_smollm3_smol_summarize_no_think', 'smoltalk_smollm3_systemchats_30k_no_think', 'table_gpt_no_think', 'tulu_3_sft_personas_instruction_following_no_think', 'xlam_traces_no_th

In [5]:
model_name = "HuggingFaceTB/SmolLM-135M"
base = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name + '-Instruct')

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [6]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

In [7]:
instucut_name = model_name + '-Instruct'
instruct = AutoModelForCausalLM.from_pretrained(instucut_name).to(device)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [8]:
print(f"Model Size: {(base.get_memory_footprint() / 1024**3):.3f}GB")

Model Size: 0.251GB


In [9]:
tokenizer.chat_template

"{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}"

In [10]:
cols_to_remove = ds['Mixture_of_Thoughts_science_no_think'].column_names
cols_to_remove

['messages', 'chat_template_kwargs', 'source']

In [11]:
def format_ds(batch):
    return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}

text = ds.map(format_ds, batched=True, remove_columns=cols_to_remove)

In [12]:
text['Mixture_of_Thoughts_science_no_think'][0]

{'text': "<|im_start|>user\nWhat hormone's action is inhibited by caffeine, leading to increased urination?A: ADH\nB: Insulin\nC: Thyroxine\nD: Cortisol<|im_end|>\n<|im_start|>assistant\nCaffeine acts as a diuretic by inhibiting the action of antidiuretic hormone (ADH), which is responsible for signaling the kidneys to reabsorb water and concentrate urine. When ADH is suppressed, the kidneys excrete more water, leading to increased urination. The other hormones listed—insulin (regulates blood sugar), thyroxine (regulates metabolism), and cortisol (involved in stress response)—are not directly related to fluid balance or diuresis. \n\n**Answer: A**  \n\\boxed{A}<|im_end|>\n"}

In [13]:
tokenizer.eos_token_id

2

In [14]:
# prompt = "I'd like a recipe for something warm."
# tokenized = tokenizer(prompt, return_tensors='pt').to(device)

# with torch.no_grad():
#   outputs = base.generate(
#       **tokenized,
#       max_new_tokens=200,
#       temperature=0.7,
#       do_sample=True,
#       pad_token_id=tokenizer.eos_token_id
#   )
#   decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
#   print(decoded)



In [15]:
base.device

device(type='cuda', index=0)

In [16]:
instruct.name_or_path

'HuggingFaceTB/SmolLM-135M-Instruct'

In [17]:
# # Sampling from the instruct model

# message = [{
#     "role": "user",
#     "content": prompt
# }]

# formatted = tokenizer.apply_chat_template(message, tokenize=False, add_generation_prompt=True)

# formatted_tokenized = tokenizer(formatted, return_tensors='pt').to(device)

# output = instruct.generate(
#     **formatted_tokenized,
#     max_new_tokens=200,
#     temperature=0.5,
#     do_sample=True,
#     pad_token_id=tokenizer.eos_token_id
# )

# decoded = tokenizer.decode(output[0])
# print(decoded)
# #

In [18]:
# assistant_start = tokenizer.bos_token + 'assistant\n'
# response_start = decoded.find(assistant_start) + len(assistant_start)
# response = decoded[response_start:].split(tokenizer.eos_token)[0]
# print(response)

### Now FT the base model

In [ ]:
# def preprocess(batch):
#   return {'text': [tokenizer.apply_chat_template(ex, tokenize=False) for ex in batch['messages']]}


In [21]:
output_dir=f"./checkpoints/{model_name}-Mine"

training_config = SFTConfig(
    # Model and data
    output_dir=output_dir,
    dataset_text_field="text",
    max_length=512,

    # Training hyperparameters
    per_device_train_batch_size=80,  # Tried mutliple batchsizes for a few iters. 160 batch size uses 12.8/15 GB.
    gradient_accumulation_steps=4, # Increased since I'm uisng my rtx and want to match the T4 config.
    learning_rate=5e-5,
    num_train_epochs=1,  # Start with 1 epoch
    # Need to estimate how many tokens the splits i'm using have to pick a good iter number.
    # Considering sequence packing, it's hard to get an accurate computation of how many batches the dataset fits.
    max_steps=2000,

    # Optimization
    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch",

    # Logging and saving
    logging_steps=10,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=100,
    eval_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,

    # Memory optimization
    dataloader_num_workers=0,
    # group_by_length=True,  # Group similar length sequences

    # Hugging Face Hub integration
    push_to_hub=False,  # Set to True to upload to Hub
    hub_model_id=f"your-username/{model_name}",

    # Experiment tracking
    # report_to=["trackio"],  # Use trackio for experiment tracking
    run_name=f"{model_name}-training",
)

print("Training configuration set!")
print(f"Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

Training configuration set!
Effective batch size: 320


In [ ]:
used_splits = [col for col in text.column_names if 'no_think' in col] # 135M doesn't have think mode in Chattemplate. only 3B model does.
used_splits
# Print a row from each split to check format.
# for split in used_splits:
#     example = next(iter(ds[split].take(1)))
#     print(f"--- {split} ---")
#     print(example)
#     print()


['LongAlign_64k_context_lang_annotated_lang_6_no_think',
 'Mixture_of_Thoughts_science_no_think',
 'OpenHermes_2.5_no_think',
 'OpenThoughts3_1.2M_no_think_no_think',
 'hermes_function_calling_v1_no_think',
 'smoltalk_multilingual_8languages_lang_5_no_think',
 'smoltalk_smollm3_everyday_conversations_no_think',
 'smoltalk_smollm3_explore_instruct_rewriting_no_think',
 'smoltalk_smollm3_smol_magpie_ultra_no_think',
 'smoltalk_smollm3_smol_rewrite_no_think',
 'smoltalk_smollm3_smol_summarize_no_think',
 'smoltalk_smollm3_systemchats_30k_no_think',
 'table_gpt_no_think',
 'tulu_3_sft_personas_instruction_following_no_think',
 'xlam_traces_no_think']

In [26]:
# base.config

In [40]:
print('==== used splits length ====\n')
split_sizes = []
for split in used_splits:
    length = len(ds[split])
    split_sizes.append(length)
    print(f'{split}:  {length}')

print('split sizes:', split_sizes)
print('min split: ', min(split_sizes))
print('max split: ', max(split_sizes))

==== used splits length ====

LongAlign_64k_context_lang_annotated_lang_6_no_think:  6249
Mixture_of_Thoughts_science_no_think:  86110
OpenHermes_2.5_no_think:  384900
OpenThoughts3_1.2M_no_think_no_think:  435193
hermes_function_calling_v1_no_think:  8961
smoltalk_multilingual_8languages_lang_5_no_think:  254047
smoltalk_smollm3_everyday_conversations_no_think:  2260
smoltalk_smollm3_explore_instruct_rewriting_no_think:  30391
smoltalk_smollm3_smol_magpie_ultra_no_think:  406843
smoltalk_smollm3_smol_rewrite_no_think:  53262
smoltalk_smollm3_smol_summarize_no_think:  96061
smoltalk_smollm3_systemchats_30k_no_think:  33997
table_gpt_no_think:  13203
tulu_3_sft_personas_instruction_following_no_think:  29970
xlam_traces_no_think:  59962
split sizes: [6249, 86110, 384900, 435193, 8961, 254047, 2260, 30391, 406843, 53262, 96061, 33997, 13203, 29970, 59962]
min split:  2260
max split:  435193


In [31]:
from datasets import interleave_datasets
eval_ex_num = 400
eval_ds = []
train_ds = []
for split in used_splits:
  s = ds[split].map(preprocess, batched=True, remove_columns=ds[split].column_names)
  eval_ds.append(list(s.take(eval_ex_num)))
  train_ds.append(s.skip(eval_ex_num))

eval_ds = [ex for part in eval_ds for ex in part] # Flatten the eval list of lists
probas = [(size - eval_ex_num) / (row_count - eval_ex_num * len(used_splits)) for size in split_sizes]

## interleave_ds here sorta orders the train ds by selection from each split based on the probas.
# This way you get more diverse examples from the start instead of exhausting each split at a time.
train_ds = interleave_datasets(train_ds, probabilities=probas, stopping_strategy="all_exhausted")


In [32]:
x = next(iter(ds[used_splits[0]]))
x

{'messages': [{'content': 'Hi there', 'role': 'user'},
  {'content': 'Hello! How can I help you today?', 'role': 'assistant'},
  {'content': "I'm looking for a healthy breakfast idea. What's a good option?",
   'role': 'user'},
  {'content': "A fruit salad is a great choice. It's nutritious, delicious, and easy to make.",
   'role': 'assistant'},
  {'content': 'What fruits are good in a fruit salad?', 'role': 'user'},
  {'content': 'You can use a mix of your favorite fruits, such as strawberries, bananas, grapes, and pineapple.',
   'role': 'assistant'}],
 'chat_template_kwargs': {'custom_instructions': '',
  'enable_thinking': False,
  'python_tools': [],
  'xml_tools': []},
 'source': 'smoltalk-smollm3_everyday-conversations'}

In [33]:
# Checking how many big one sequence is.
# Does TRL handle if a tokenized row produces more tokens than sequence_len?
tokenized_example = tokenizer.apply_chat_template(x['messages'], tokenize=True)
tokenized_example['input_ids'].__len__()

108

In [34]:
eval_ds[0]

{'text': "<|im_start|>user\nHi there<|im_end|>\n<|im_start|>assistant\nHello! How can I help you today?<|im_end|>\n<|im_start|>user\nI'm looking for a healthy breakfast idea. What's a good option?<|im_end|>\n<|im_start|>assistant\nA fruit salad is a great choice. It's nutritious, delicious, and easy to make.<|im_end|>\n<|im_start|>user\nWhat fruits are good in a fruit salad?<|im_end|>\n<|im_start|>assistant\nYou can use a mix of your favorite fruits, such as strawberries, bananas, grapes, and pineapple.<|im_end|>\n"}

In [35]:
from datasets import Dataset
eval_ds = Dataset.from_list(eval_ds)

In [36]:

trainer = SFTTrainer(
    model=base,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=training_config,
    processing_class=tokenizer,
)

Adding EOS to eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2993 > 2048). Running this sequence through the model will result in indexing errors


Building labels for eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss
